# Quick start: the gate on one LIBERO-Spatial scene

This notebook runs the verification layer on one scene of LIBERO-Spatial (task 0, initial state 5) and shows its decision for a
correct instruction, an impossible instruction, and all seven test conditions of the paper.

* **Time:** about 3-5 minutes. The first run downloads Grounding DINO (about 0.9 GB) and OWLv2 (about 0.6 GB).
* **Needs:** the environment from the README (LeRobot with LIBERO, a CUDA GPU). The optional last part also downloads SmolVLA.
* The gate never moves the robot and never changes the policy's actions. It only answers: may the policy act on this instruction?

In [ ]:
import os, sys
os.environ.setdefault("MUJOCO_GL", "egl")             # off-screen rendering (Linux / WSL)
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
sys.path.insert(0, os.path.abspath(".."))             # lets this notebook import vla_gate without installing it

import matplotlib.pyplot as plt
from vla_gate import SpatialGate, flipped_image
from vla_gate.sim import KINDS, make_case, make_env, perturb_instruction, setup_episode
from vla_gate.viz import show_result

## 1. Load the gate

`"hybrid"` is the gate of Study 2: OWLv2 grounds the bowls, the ramekin and the plate; Grounding DINO grounds the cabinet, the
stove and the cookie box. The weights and thresholds are the frozen ones of the paper:
**C = 0.27 G + 0.40 S + 0.33 D**, execute if C ≥ 0.70, otherwise re-ground once, stop if C < 0.50.
(`SpatialGate.load("gdino")` gives the first gate of Study 1.)

In [ ]:
gate = SpatialGate.load("hybrid")
print("gate ready | weights", gate.cfg["weights"][:3], "| execute if C >=", gate.cfg["high"], "| stop if C <", gate.cfg["low"])

## 2. A LIBERO-Spatial scene

Task 0 asks for the black bowl *between the plate and the ramekin*. Both black bowls look the same, so only the spatial relation
tells them apart.

In [ ]:
TASK, STATE = 0, 5
env, task, instruction = make_env(TASK)
obs = setup_episode(env, TASK, STATE)
print("instruction:", instruction)
plt.figure(figsize=(4, 4)); plt.imshow(flipped_image(obs)); plt.axis("off"); plt.show()

## 3. The correct instruction

G = grounding (were the named objects found?), S = the relation in the image, D = the relation in 3D (with depth).
Boxes: red = the bowl the gate selected, cyan = reference, green = goal, dashed yellow = the other bowl candidates.

In [ ]:
result = gate.check_libero(env, obs, instruction)
print(result)
show_result(flipped_image(obs), result, "correct instruction")

## 4. An impossible instruction: the object to pick up is not on the table

SmolVLA alone would still move (it acts on every instruction). The gate finds no blue teapot, so G = 0 and it stops the robot.

In [ ]:
wrong = perturb_instruction(instruction, "absent_target", STATE)       # 'blue teapot' instead of 'black bowl'
obs = setup_episode(env, TASK, STATE, instruction=wrong)
result = gate.check_libero(env, obs, wrong)
print("instruction:", wrong)
print(result)
show_result(flipped_image(obs), result, "impossible instruction")

## 5. All seven test conditions on this scene

Three conditions make the instruction impossible (absent reference, absent target, relation violated: the bowl was moved away);
four keep it valid (nominal, an extra distractor bowl, a moved plate, a grey patch over the reference).

The last column is the decision of our attached run (Table VII of the paper) for these same scenes, so you can check that your
installation reproduces it. On this scene the gate also makes two mistakes: it lets the relation violation through (all 8
impossible instructions the hybrid gate released on the 350 test scenes were relation violations) and it refuses the valid
moved-plate scene.

In [ ]:
OUR_RUN = {"clean": ("EXECUTE", 1.00), "absent_reference": ("ABSTAIN", 0.00), "absent_target": ("ABSTAIN", 0.00),
           "relation_violation": ("EXECUTE", 0.60), "distractor": ("EXECUTE", 1.00), "position_change": ("ABSTAIN", 0.29),
           "occlusion": ("EXECUTE", 0.97)}                     # hybrid gate, task 0, initial state 5 (attached run)
print(f"{'condition':19s} {'valid?':7s} {'G':>5s} {'S':>5s} {'D':>5s} {'C':>5s}  {'decision':9s} our run")
n_same = 0
for kind in KINDS:
    obs, instr = make_case(env, TASK, STATE, kind)
    r = gate.check_libero(env, obs, instr)
    exp = OUR_RUN[kind]
    same = r.decision == exp[0] and abs(r.C - exp[1]) < 0.006         # our log printed C with two decimals
    n_same += same
    valid = "valid" if kind in ("clean", "distractor", "position_change", "occlusion") else "WRONG"
    print(f"{kind:19s} {valid:7s} {r.G:5.2f} {r.S:5.2f} {r.D:5.2f} {r.C:5.2f}  {'EXECUTE' if r.execute else 'STOP':9s} "
          f"{exp[0].replace('ABSTAIN', 'STOP')} (C={exp[1]:.2f}) {'same' if same else '<- DIFFERENT'}")
print(f"\n{n_same}/7 decisions identical to our run")

## 6. Optional: let SmolVLA act when the gate says EXECUTE (about 2 minutes)

Task 1, initial state 5: the gate releases the correct instruction and SmolVLA completes it (in our attached run as well).
The gate is checked first; only after EXECUTE is the policy run on the same scene.

In [ ]:
from vla_gate.policies import SmolVLA, rollout_smolvla

policy = SmolVLA()                                         # HuggingFaceVLA/smolvla_libero
env1, _, instruction1 = make_env(1)
obs, instr = make_case(env1, 1, 5, "clean", policy=policy)
check = gate.check_libero(env1, obs, instr)
print(instr); print(check)
frames = []
if check.execute:
    episode = rollout_smolvla(env1, policy, 1, 5, instr, on_frame=lambda o: frames.append(flipped_image(o)))
    print(f"SmolVLA: success = {episode['success']} after {episode['episode_length']} steps ({episode['failure_reason']})")
    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    for a, i in zip(ax, [0, len(frames) // 3, 2 * len(frames) // 3, len(frames) - 1]):
        a.imshow(frames[i]); a.set_title(f"step {i}"); a.axis("off")
    plt.show()
env1.close()

In [ ]:
env.close()

## Next

* `experiments/` holds the exact code behind every table of the paper (Study 1, Study 2, the second policy and the VLM judge,
  the attached run and the demo video).
* `python scripts/reproduce_numbers.py` recomputes the paper's main numbers from the saved result files in seconds.
* In your own code: `result = gate.check(image, depth, camera, instruction)` works on any RGB-D camera image; see `vla_gate/gate.py`.